# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?


In [8]:
# 여기에 코드를 작성하세요.

# 1. user_id 기준 중복 여부
print('[user_id 중복값 개수] :', df['userid'].duplicated().sum())

# 2. version의 고유값, 그룹별 사용자 수
print('\n[version의 고유값] :', df['version'].unique())

print('\n[version별 사용자 수]')
display(df.groupby('version')[['userid']].count().reset_index())


# 3. 버전별 요약표
summary = (df.groupby("version").agg(
          사용자수=("userid", "nunique"),
          평균_게임_라운드=("sum_gamerounds", "mean"),
          유지율_1일=("retention_1", "mean"),
          유지율_7일=("retention_7", "mean")
      ).reset_index()
)

print('\n[요약표]')
display(summary.round(3))

[user_id 중복값 개수] : 0

[version의 고유값] : <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

[version별 사용자 수]


,version,userid
0,gate_30,44700
1,gate_40,45489



[요약표]


,version,사용자수,평균_게임_라운드,유지율_1일,유지율_7일
0,gate_30,44700,52.456,0.448,0.190
1,gate_40,45489,51.299,0.442,0.182


### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
    - userid로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
    - 기존 게이트 위치인 gate_30을 대조군, 게이트를 40단계로 옮긴 gate_40을 실험군으로 설정

- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
    - 핵심지표는 단순 1일 평가보다는 7일 이후 시점을 나타내는 retation_7로(재방문)
    - 보조지표는 초기 반응 확인을 위해 retatation_1로
    - 가드레일 지표는 활동이 감소하는지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수)로 본다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
    - 아닙니다. 표본 변동으로 생길 수 있는 차이인지? 통계적으로 검증이 필요하다.

---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?


In [9]:
# 여기에 코드를 작성하세요.

rng = np.random.default_rng(seed=42)

randomized_users = df[["userid"]].drop_duplicates().copy()

randomized_users["group"] = rng.choice(["A", "B"], size=len(randomized_users), p=[0.5, 0.5])

print('[ 무작위 배정 결과 ]')
display(randomized_users.head())

# 연습용 A/B 그룹별 인원수와 비율
randomized_summary = randomized_users["group"].value_counts().reset_index()

randomized_summary["ratio"] = randomized_summary["count"] / randomized_summary["count"].sum()

randomized_summary["ratio_pct"] = (randomized_summary["ratio"] * 100).round(2)

print("\n[ 무작위 배정 기술통계 ]")
display(randomized_summary[["group", "count", "ratio_pct"]].sort_values('group'))



[ 무작위 배정 결과 ]


,userid,group
0,116,B
1,337,A
2,377,B
3,483,B
4,488,A



[ 무작위 배정 기술통계 ]


,group,count,ratio_pct
1,A,44852,49.73
0,B,45337,50.27


### 질문

- 시드를 고정하는 이유는 무엇인가요?
    - 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻도 코드를 확인할 수 있게 하기 위해서

- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
    - userid별로 practice_group의 고유값 수를 계산하여 최대값이 1인지 확인한다.

- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
    - p-value가 0.05보다 작으므로, 50:50 계획과 통계적으로 일치한다고 보기 어렵다.

- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
    - 세 변수는 버전을 경험한 이후의 측정된 결과이므로 처치영향을 받을 수 있다.
    - 균형점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인 같은 공변량이 필요하지만 현 데이터에서는 제공되지 않았다.

---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


#### **A/B Test 계획서**

이 A/B 테스트의 목적은 기존 버전인 `gate_30`과 새로운 버전인 `gate_40`을 비교하여,    
새로운 버전이 사용자의 장기적인 게임 이용 유지에 어떤 영향을 미치는지 확인하는 것이다.

> 1. 가설 설정

- `귀무가설` : 기존 버전(gate_30)과 새 버전(gate_40)의 설치 후 7일 재방문율(retention_7)에는 차이가 없다.

- `대립가설` : 기존 버전(gate_30)과 새 버전(gate_40)의 설치 후 7일 재방문율(retention_7)에는 차이가 있다.

<br>


> 2. 실험 설계

- `실험 단위` : 개별 사용자(`userid`)

- `대조군` : 기존 버전인 `gate_30`

- `실험군` : 새로운 버전인 `gate_40`

    - 대조군과 실험군에 배정시에는 사용자를 두 그룹에 무작위로 배정하고, 동일한 사용자가 두 버전에 중복 배정되지 않도록 한다.

- `핵심 지표` : 설치 후 7일 재방문율(`retention_7`)

    - 새로운 버전이 사용자의 장기적인 게임 이용 유지에 미치는 영향을 확인한다.

- `보조 지표` : 설치 후 1일 재방문율(`retention_1`)
    - 버전 변경이 사용자의 초기 이용 경험에 미치는 영향을 확인한다.

- `가드레일 지표` : 게임 진행 라운드(`sum_gamerounds`)
    - 7일 재방문율이 좋아져도 실제 게임 이용량이 크게 감소하는 현상이 있는지 확인한다.

<br>

> 3. 실험 실행

- 사용자가 실험에 처음 참여할 때 `gate_30` 또는 `gate_40` 둘 중 하나에 무작위로 배정하고, 실험 기간 동안 동일한 버전을 유지해야한다.

- 실험 전에는 사용자 중복여부, 그룹별 표본 수가 정당한지를 확인해야한다.

- 그리고 수집해야 할 데이터의 정보, 데이터 타입 등을 정해야한다.

<br>

> 4. 결과분석   

`gate_30`과 `gate_40`의 버전별 관측 지표를 비교하여 사용자 수, 1일 재방문율, 7일 재방문율, 게임 진행 라운드 간의 차이를 

확인하였다. 7일 재방문률을 핵심지표로 선정하였으며, 현재 수집한 데이터는 각 버전에서 수집된 관측값으로 두 버전 사이에 차이가 

존재한다는 것을 알 수 있다. 하지만 이 관측치는 새로운 버전의 효과로 인해 발생한건지 다른 외부변수에 영향을 받아 차이가 발생한건지는 

알 수 없다. 따라서 새 버전에서 7일 재방문율 값이 높다고 하더라도 새로운 버전을 바로 더 좋은 버전이라고 결론 내리기는 어렵다. 

적절한 통계적 검정을 통해 살펴보아야하는 부분이다. 따라서 현재 `gate_30` 유지 또는 `gate_40` 전환에 대한 최종 의사결정을 

보류하고, 통계적 검정 결과까지 확인한 뒤 최종으로 이어져야 한다.



---

## 실습 마무리

- 어떤 문제가 있었는가?
    - 그룹별 유지율과 평균 게임 라운드만 비교하여 사용자의 고정배정 여부, 계획한 표본 비율, 지표 사전 지정, 
    - 외부간섭, 로깅 등의 문제를 놓칠 수 있었다.

- 어떻게 개선했는가?
    - 실험단위를 userid로 명시하도 대조군, 실험군과 3가지 지표를 사전에 정의했다.
    - 실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과변수, 사전 공변량의 차이 등을 함께 점검했다.

- 무엇을 근거로 개선되었다고 판단했는가?
    - 사용자 중복0건, 실제 그룹비율 약 49.5%, 50.4%, 적합도 검정 p-value가 0.05 미만을 확인하여 단순히 비슷해 보인다는 필수 1의 판단보다는 구체적인 조사 증거를 확보했다.
    - 40gate를 배포 전에 기술통계를 통해 버전 업 배포 결정을 보류했다.

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
